# Article 4: Single-Agent Benchmark Analysis

This notebook analyzes benchmark results comparing ReAct vs Plan-and-Execute agent architectures on multi-tool queries, measuring success rate, latency, tool usage patterns, and error recovery behavior.

Teaching note: Agent architecture comparison
--------------------------------------------
Effective agent benchmarking requires measuring:
1. **Success rate**: Did the agent complete the task?
2. **Tool efficiency**: How many tools were called? Which ones?
3. **Latency**: Total time including LLM calls + tool execution
4. **Error patterns**: What failure modes exist?

Key architectural differences:
- **ReAct**: Iterative reasoning loop (think → act → observe) - adapts dynamically
- **Plan-Execute**: Upfront planning + sequential execution - fewer LLM calls if plan is good

Neither is universally better - depends on query complexity and failure modes.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set style for production-quality charts
# Teaching note: Chart aesthetics
# - seaborn whitegrid: Clean, professional look
# - Figure size (10, 6): Readable in blog posts
# - DPI 300: Publication quality for PNG export
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["font.size"] = 11

## Load Benchmark Results

Load results from `results/data/article_04_benchmarks.json` generated by `benchmarks/run_article_04.py`.

In [ ]:
# Define paths
# Teaching note: When running from notebooks/, go up one level to project root
PROJECT_ROOT = Path("..").resolve()
RESULTS_DIR = PROJECT_ROOT / "results" / "data"
CHARTS_DIR = PROJECT_ROOT / "results" / "charts" / "article_04"

# Ensure output directory exists
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

# Load benchmark data
benchmark_file = RESULTS_DIR / "article_04_benchmarks.json"

if not benchmark_file.exists():
    raise FileNotFoundError(
        f"Benchmark file not found: {benchmark_file}\n"
        "Run: uv run python benchmarks/run_article_04.py --mock-tools"
    )

with open(benchmark_file) as f:
    data = json.load(f)

print(f"Loaded benchmark data from {benchmark_file}")
print(f"Number of runs: {data['metadata']['runs']}")
print(f"Total queries: {data['metadata']['total_queries']}")
print(f"Mock mode: {data['metadata']['use_mock']}")
print(f"Timestamp: {data['metadata']['timestamp']}")

## Extract Metrics

Extract summary metrics for both agent types.

In [ ]:
# Extract summaries
react_summary = data["summaries"]["react"]
plan_summary = data["summaries"]["plan_execute"]

# Extract detailed results for distribution plots
react_results = data["detailed_results"]["react"]
plan_results = data["detailed_results"]["plan_execute"]

print("ReAct Agent:")
print(f"  Success Rate: {react_summary['success_rate']:.1%}")
print(f"  Avg Latency: {react_summary['avg_latency_ms']:.0f}ms")
print(f"  Median Latency: {react_summary['median_latency_ms']:.0f}ms")
print(f"  Avg Tool Calls: {react_summary['avg_tool_calls']:.1f}")
print(f"  Errors: {react_summary['error_count']}")
print()
print("Plan-and-Execute Agent:")
print(f"  Success Rate: {plan_summary['success_rate']:.1%}")
print(f"  Avg Latency: {plan_summary['avg_latency_ms']:.0f}ms")
print(f"  Median Latency: {plan_summary['median_latency_ms']:.0f}ms")
print(f"  Avg Tool Calls: {plan_summary['avg_tool_calls']:.1f}")
print(f"  Errors: {plan_summary['error_count']}")

## Chart 1: Success Rate Comparison

Bar chart comparing task completion success rates between agent architectures.

Teaching note: Success rate interpretation
-------------------------------------------
- **100% success**: Agent completes all queries without errors
- **High success (80-99%)**: Agent is reliable but has edge cases
- **Low success (<80%)**: Agent struggles with query complexity or tool failures

Success rate differences reveal:
- Planning failures (Plan-Execute with bad upfront plan)
- Adaptation failures (ReAct unable to recover from tool errors)
- Architectural strengths/weaknesses on specific query types

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Data for plotting
agents = ["ReAct", "Plan-and-Execute"]
success_rates = [
    react_summary["success_rate"] * 100,
    plan_summary["success_rate"] * 100
]
error_counts = [
    react_summary["error_count"],
    plan_summary["error_count"]
]

x = np.arange(len(agents))
width = 0.6

# Create bars
bars = ax.bar(x, success_rates, width, color=['#3498db', '#2ecc71'])

# Add error count annotations
for i, (bar, err_count) in enumerate(zip(bars, error_counts)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%\n({err_count} errors)',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Formatting
ax.set_xlabel('Agent Architecture', fontweight='bold')
ax.set_ylabel('Success Rate (%)', fontweight='bold')
ax.set_title('Task Completion Success Rate', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(agents)
ax.set_ylim(0, 110)  # Extra space for labels
ax.grid(axis='y', alpha=0.3)

# Add reference line at 100%
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5, label='100% success')
ax.legend(loc='lower right')

plt.tight_layout()
output_file = CHARTS_DIR / "01_success_rate.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"Saved: {output_file}")
plt.show()

## Chart 2: Tool Usage Histogram

Compare which tools each agent architecture used and how frequently.

Teaching note: Tool usage patterns
-----------------------------------
Tool usage reveals agent behavior:
- **High tool usage**: Agent actively uses available tools (good for complex queries)
- **Low tool usage**: Agent relies more on parametric knowledge (risky for factual queries)
- **Tool distribution**: Which tools are most valuable? Are some never used?

Architecture differences:
- ReAct may call tools iteratively (higher total calls)
- Plan-Execute may call tools in parallel (efficient if planned well)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Extract tool usage
react_tools = react_summary.get("tool_usage_histogram", {})
plan_tools = plan_summary.get("tool_usage_histogram", {})

# Get all unique tools
all_tools = sorted(set(list(react_tools.keys()) + list(plan_tools.keys())))

# Prepare data
react_counts = [react_tools.get(tool, 0) for tool in all_tools]
plan_counts = [plan_tools.get(tool, 0) for tool in all_tools]

x = np.arange(len(all_tools))
width = 0.35

# Create grouped bars
bars1 = ax.bar(x - width/2, react_counts, width, label='ReAct', color='#3498db')
bars2 = ax.bar(x + width/2, plan_counts, width, label='Plan-and-Execute', color='#2ecc71')

# Formatting
ax.set_xlabel('Tool Name', fontweight='bold')
ax.set_ylabel('Total Calls', fontweight='bold')
ax.set_title('Tool Usage Histogram', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(all_tools, rotation=45, ha='right')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars if there are calls
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
output_file = CHARTS_DIR / "02_tool_usage.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"Saved: {output_file}")
plt.show()

## Chart 3: Latency Distribution

Box plot comparing end-to-end query latency distributions between agent architectures.

Teaching note: Latency comparison
----------------------------------
Latency differences reveal architectural trade-offs:

**ReAct**:
- Pro: Can stop early if answer is found
- Con: Iterative LLM calls add overhead
- Expected: Higher variance (adapts based on query complexity)

**Plan-Execute**:
- Pro: Predictable execution (plan once, execute steps)
- Con: Cannot skip unnecessary steps in plan
- Expected: Lower variance (fixed plan structure)

Median latency = typical user experience
Wide IQR = unpredictable performance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Extract latency data from detailed results
react_latencies = [r["latency_ms"] for r in react_results]
plan_latencies = [r["latency_ms"] for r in plan_results]

# Prepare data for box plot
latency_data = [react_latencies, plan_latencies]
agent_labels = ["ReAct", "Plan-and-Execute"]

# Create box plot
bp = ax.boxplot(latency_data, labels=agent_labels, patch_artist=True,
                showmeans=True, meanline=True,
                boxprops=dict(alpha=0.7),
                medianprops=dict(color='red', linewidth=2),
                meanprops=dict(color='green', linewidth=2, linestyle='--'),
                whiskerprops=dict(color='#34495e'),
                capprops=dict(color='#34495e'))

# Color the boxes
colors = ['#3498db', '#2ecc71']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

# Formatting
ax.set_xlabel('Agent Architecture', fontweight='bold')
ax.set_ylabel('Latency (ms)', fontweight='bold')
ax.set_title('Query Latency Distribution', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add legend for median and mean
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='red', linewidth=2, label='Median'),
    Line2D([0], [0], color='green', linewidth=2, linestyle='--', label='Mean')
]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
output_file = CHARTS_DIR / "03_latency_distribution.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"Saved: {output_file}")
plt.show()

## Chart 4: Iterations/Steps vs Latency

Scatter plot showing relationship between agent execution complexity (iterations for ReAct, steps for Plan-Execute) and latency.

Teaching note: Execution complexity
------------------------------------
This chart reveals:
- **Linear relationship**: More iterations/steps → higher latency (expected)
- **Outliers**: Queries with disproportionate latency (tool failures, LLM delays)
- **Efficiency**: Which agent completes queries with fewer LLM round-trips?

ReAct iterations vs Plan-Execute steps:
- ReAct: Each iteration = LLM call + optional tool call + observation
- Plan-Execute: Steps are pre-planned, executed sequentially

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Extract iterations and steps
react_iterations = [r.get("iterations", 0) for r in react_results]
react_latencies_for_scatter = [r["latency_ms"] for r in react_results]

plan_steps = [r.get("steps", 0) for r in plan_results]
plan_latencies_for_scatter = [r["latency_ms"] for r in plan_results]

# Plot ReAct
ax.scatter(react_iterations, react_latencies_for_scatter,
           s=100, alpha=0.6, color='#3498db', marker='o',
           label='ReAct (iterations)', edgecolors='white', linewidth=1.5)

# Plot Plan-Execute
ax.scatter(plan_steps, plan_latencies_for_scatter,
           s=100, alpha=0.6, color='#2ecc71', marker='s',
           label='Plan-Execute (steps)', edgecolors='white', linewidth=1.5)

# Formatting
ax.set_xlabel('Iterations / Steps', fontweight='bold')
ax.set_ylabel('Latency (ms)', fontweight='bold')
ax.set_title('Execution Complexity vs Latency', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', framealpha=0.9)

plt.tight_layout()
output_file = CHARTS_DIR / "04_complexity_vs_latency.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"Saved: {output_file}")
plt.show()

## Summary Statistics Table

Create a summary table comparing all key metrics.

In [ ]:
summary_data = {
    "Agent": ["ReAct", "Plan-and-Execute"],
    "Success Rate": [
        f"{react_summary['success_rate']:.1%}",
        f"{plan_summary['success_rate']:.1%}"
    ],
    "Avg Latency (ms)": [
        f"{react_summary['avg_latency_ms']:.0f}",
        f"{plan_summary['avg_latency_ms']:.0f}"
    ],
    "Median Latency (ms)": [
        f"{react_summary['median_latency_ms']:.0f}",
        f"{plan_summary['median_latency_ms']:.0f}"
    ],
    "Avg Tool Calls": [
        f"{react_summary['avg_tool_calls']:.1f}",
        f"{plan_summary['avg_tool_calls']:.1f}"
    ],
    "Errors": [
        str(react_summary['error_count']),
        str(plan_summary['error_count'])
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\nSummary Statistics:")
print("=" * 100)
print(summary_df.to_string(index=False))
print("=" * 100)

## Results Interpretation

### Agent Architecture Comparison

This benchmark establishes performance characteristics for single-agent architectures on multi-tool queries.

**Key Observations:**

1. **Success Rate**: Measures task completion reliability
   - High success = robust error recovery
   - Low success = struggles with query complexity or tool failures

2. **Latency**: End-to-end response time
   - ReAct: Adaptive (stops when done, but iterative LLM calls add overhead)
   - Plan-Execute: Predictable (fixed plan structure)

3. **Tool Usage**: Reveals agent behavior patterns
   - Which tools are essential vs underutilized?
   - Does agent prefer parametric knowledge over tool calls?

4. **Execution Complexity**: Iterations/steps vs latency relationship
   - Linear = expected behavior
   - Outliers = tool failures or LLM delays

**Next Steps:**

Article 5 will implement multi-agent orchestration (LangGraph, CrewAI, AutoGen) to compare collaborative architectures against these single-agent baselines.

## Export Complete

All charts have been exported to `results/charts/article_04/`:
- `01_success_rate.png`: Success rate comparison bar chart
- `02_tool_usage.png`: Tool usage histogram
- `03_latency_distribution.png`: Latency box plot
- `04_complexity_vs_latency.png`: Iterations/steps vs latency scatter plot